In [1]:
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import classification_report, confusion_matrix

from datasets import Dataset
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments
)

/Users/valtervalenti/artificial-intelligence/deep-learning/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_csv_path = "../data/imdb/train.csv"
test_csv_path = "../data/imdb/test.csv"

print("Carico dataset dai CSV locali...")

train_df = pd.read_csv(train_csv_path)
test_df = pd.read_csv(test_csv_path)

train_texts = train_df["text"].tolist()
train_labels = train_df["label"].tolist()
test_texts = test_df["text"].tolist()
test_labels = test_df["label"].tolist()

print("Train size:", len(train_texts), 
      "Test size:", len(test_texts))

Carico dataset dai CSV locali...
Train size: 25000 Test size: 25000


In [3]:
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [4]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [5]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

In [6]:
train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])

train_dataset = train_dataset.rename_column("label", "labels")
test_dataset = test_dataset.rename_column("label", "labels")

train_dataset.set_format("torch")
test_dataset.set_format("torch")

Map: 100%|██████████| 25000/25000 [00:04<00:00, 5562.01 examples/s]


In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [8]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1374.79it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those p

In [9]:
model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [10]:
training_args = TrainingArguments(
    output_dir="../data/imdb/bert",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01
)

In [11]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

In [12]:
trainer.train()

/Users/valtervalenti/artificial-intelligence/deep-learning/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
500,0.340109
1000,0.258252
1500,0.244939
2000,0.170980
2500,0.155569
3000,0.152298
3500,0.104819
4000,0.088479
4500,0.081898


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.78it/s]
/Users/valtervalenti/artificial-intelligence/deep-learning/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.71it/s]
/Users/valtervalenti/artificial-intelligence/deep-learning/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.27it/s]


TrainOutput(global_step=4689, training_loss=0.1738103230794271, metrics={'train_runtime': 8545.7804, 'train_samples_per_second': 8.776, 'train_steps_per_second': 0.549, 'total_flos': 9866664576000000.0, 'train_loss': 0.1738103230794271, 'epoch': 3.0})

In [13]:
print("\nCalcolo predizioni sul test set...")

predictions = trainer.predict(test_dataset)

logits = predictions.predictions
pred_labels = np.argmax(logits, axis=1)
true_labels = predictions.label_ids


Calcolo predizioni sul test set...


/Users/valtervalenti/artificial-intelligence/deep-learning/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


In [14]:
accuracy = np.mean(pred_labels == true_labels)
print(f"\nAccuracy finale: {accuracy:.4f}")


Accuracy finale: 0.9241


In [15]:
bert_cm = confusion_matrix(true_labels, pred_labels)
print("\nConfusion Matrix:")
print(bert_cm)


Confusion Matrix:
[[11425  1075]
 [  823 11677]]


In [16]:
print("\nBERT Report")
print(classification_report(true_labels, pred_labels))


BERT Report
              precision    recall  f1-score   support

           0       0.93      0.91      0.92     12500
           1       0.92      0.93      0.92     12500

    accuracy                           0.92     25000
   macro avg       0.92      0.92      0.92     25000
weighted avg       0.92      0.92      0.92     25000

